# Product Delivery Cost Prediction

**Tabular Regression Project** — Predict last-mile delivery cost from package, route, and logistics features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `delivery_cost`

## 2 · Project Overview

We predict **last-mile delivery cost** from package characteristics, route details, and logistics parameters. Delivery cost prediction is essential for e-commerce pricing, logistics optimization, and shipping rate estimation.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given a package's weight, dimensions, delivery distance, speed tier (standard/express/same-day), fragility, zone (urban/suburban/rural), fuel price, route stops, warehouse type, and delivery time window, predict the **delivery cost in USD**.

## 5 · Why This Project Matters

- **E-commerce profitability** depends on accurate shipping cost estimation.
- **Dynamic pricing** adjusts delivery fees based on real-time cost predictions.
- **Route optimization** uses cost models to balance speed vs. expense.
- Understanding cost drivers helps negotiate better **carrier contracts**.
- Last-mile delivery is typically 50%+ of total shipping cost.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | 10 (weight, dimensions, distance, delivery type, fragile, zone, fuel price, stops, warehouse, time window) |
| **Target** | `delivery_cost` (continuous, USD) |
| **Categorical** | delivery_type (3), delivery_zone (3), warehouse_type (3) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "delivery_cost"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8000
weight_kg = np.random.lognormal(0.5, 0.8, n).clip(0.1, 100)
dimensions_cm3 = np.random.lognormal(7, 1, n).clip(100, 500000)
distance_km = np.random.lognormal(2.5, 1, n).clip(1, 500)
delivery_type = np.random.choice(["standard", "express", "same_day"], n, p=[0.5, 0.35, 0.15])
is_fragile = np.random.choice([0, 1], n, p=[0.8, 0.2])
zone = np.random.choice(["urban", "suburban", "rural"], n, p=[0.5, 0.35, 0.15])
fuel_price = np.random.uniform(3.0, 6.0, n)
num_stops = np.random.randint(1, 30, n)
warehouse = np.random.choice(["central", "regional", "local"], n, p=[0.3, 0.45, 0.25])
time_window_hours = np.random.choice([2, 4, 8, 12], n, p=[0.1, 0.25, 0.4, 0.25])

type_factor = np.where(delivery_type == "same_day", 8,
              np.where(delivery_type == "express", 4, 0)).astype(float)
zone_factor = np.where(zone == "rural", 3,
              np.where(zone == "suburban", 1, 0)).astype(float)

cost = (weight_kg * 0.8 + np.cbrt(dimensions_cm3) * 0.3
        + distance_km * 0.15 + type_factor
        + is_fragile * 3 + zone_factor
        + fuel_price * 1.5 + num_stops * 0.2
        - time_window_hours * 0.1
        + np.random.normal(0, 1.5, n))
cost = np.maximum(cost, 2)

df = pd.DataFrame({
    "weight_kg": weight_kg, "dimensions_cm3": dimensions_cm3.astype(int),
    "distance_km": distance_km, "delivery_type": delivery_type,
    "is_fragile": is_fragile, "delivery_zone": zone,
    "fuel_price_per_gallon": fuel_price, "num_route_stops": num_stops,
    "warehouse_type": warehouse, "time_window_hours": time_window_hours,
    "delivery_cost": cost
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["weight_kg", "distance_km", "dimensions_cm3",
                          "fuel_price_per_gallon", "num_route_stops", "time_window_hours"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["delivery_cost"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("delivery_cost"); ax.set_title(f"Cost vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("delivery_type")["delivery_cost"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Mean Cost by Delivery Type"); axes[0].set_xlabel("$")
df.groupby("delivery_zone")["delivery_cost"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Mean Cost by Delivery Zone"); axes[1].set_xlabel("$")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `delivery_cost`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("Cost ($)")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: ${df[TARGET].mean():.2f}, Median: ${df[TARGET].median():.2f}")
print(f"Std: ${df[TARGET].std():.2f}, Range: ${df[TARGET].min():.2f} - ${df[TARGET].max():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["volumetric_weight"] = X_train["dimensions_cm3"] / 5000
X_test["volumetric_weight"] = X_test["dimensions_cm3"] / 5000

X_train["billable_weight"] = np.maximum(X_train["weight_kg"], X_train["volumetric_weight"])
X_test["billable_weight"] = np.maximum(X_test["weight_kg"], X_test["volumetric_weight"])

X_train["fuel_distance_cost"] = X_train["fuel_price_per_gallon"] * X_train["distance_km"] / 10
X_test["fuel_distance_cost"] = X_test["fuel_price_per_gallon"] * X_test["distance_km"] / 10

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Delivery type** is the strongest cost differentiator: same-day costs ~$8 more than standard.
- **Fuel price × distance** drives the variable cost component.
- **Weight** and **dimensions** determine the billable weight (whichever is larger).
- **Rural zones** cost ~$3 more than urban — sparse deliveries have higher per-stop costs.
- **Fragile items** add ~$3 for special handling.

**Business takeaway:** Offer delivery speed tiers with pricing that reflects the true cost differential. Consolidate rural deliveries to reduce per-stop costs. Use billable weight for accurate shipping rates.

## 26 · Limitations

1. Synthetic data — real logistics costs depend on carrier contracts, fuel surcharges, and dimensional weight rules.
2. No temporal pattern — fuel prices and demand vary seasonally.
3. No failed delivery attempts — re-delivery doubles costs.
4. No returns handling — reverse logistics adds significant cost.
5. Route optimization effects (multi-stop efficiency) are simplified.

## 27 · How to Improve This Project

1. Use real carrier rate cards and shipment data.
2. Add vehicle type (van, truck, bike) as a feature.
3. Include failed delivery rate and address quality scores.
4. Model returns cost as a separate prediction.
5. Add real-time traffic and weather for same-day delivery costing.

## 28 · Production Considerations

- Integrate with order management systems for real-time cost estimation.
- Update fuel price data daily.
- Monitor prediction accuracy by carrier and region.
- Use for dynamic delivery fee pricing at checkout.

## 29 · Common Mistakes

1. Using raw weight without considering dimensional (volumetric) weight.
2. Treating all delivery types with the same cost model.
3. Ignoring fuel price volatility — static models underperform when gas prices spike.
4. Not accounting for route density (urban routes batch more deliveries per mile).
5. Assuming linear distance-cost relationship — route efficiency varies.

## 30 · Mini Challenge / Exercises

1. Remove `delivery_type` — how much does RMSE increase?
2. Compare actual weight vs. volumetric weight — which is billable more often?
3. Add a `distance × fuel_price` interaction — does it improve R²?
4. Train only on standard deliveries — does the model work for express?
5. What fuel price increase would raise average delivery cost by 10%?

## 31 · Final Summary / Key Takeaways

1. **Delivery type** (standard/express/same-day) is the top cost driver.
2. **Fuel price × distance** captures the variable logistics cost.
3. **Billable weight** (max of actual, volumetric) is a key feature in real shipping.
4. **Zone** and **fragility** add predictable surcharges.
5. **Gradient boosting** captures the non-linear delivery type × distance interactions.
6. **Real deployment** needs real-time fuel prices and carrier rate integration.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))